# OLAP · 02 · BQ2 · Is loss per event rising over time?
**Owner: M** · Reviewer: Ar · Decision supported: **whether premiums need a trend loading** (Head of Underwriting)

| Query | Question | OLAP operation |
|---|---|---|
| 2.1 | Events, loss and loss per event by year, nominal vs normalised | **Roll-up** to calendar year |
| 2.2 | Same by Australian financial year, with a 3-year moving average | **Roll-up** to an alternative hierarchy + window |
| 2.3 | What happened inside the worst year? | **Drill-down** year → quarter → month → event |
| 2.4 | Did the source stop recording small events? | **Slice** by year, distribution statistics |
| 2.5 | Like-for-like trend using only large events | **Slice** on a loss threshold, then roll-up to period |

Loss per event is **non-additive**: it is always `sum(loss) / count(events with a loss)`, never an average of averages.

In [ ]:
%run ./00_olap_setup

### 2.1 · Roll-up to calendar year
The median is shown beside the mean because one very large event can move the mean on its own.

In [ ]:
q21 = olap("bq2_by_year", """
SELECT calendar_year, count(*) AS events, count(loss_m) AS events_with_loss,
       round(sum(nominal_loss_m), 1)                 AS nominal_loss_m,
       round(sum(loss_m), 1)                         AS normalised_loss_m,
       round(sum(nominal_loss_m) / count(nominal_loss_m), 1) AS nominal_per_event_m,
       round(sum(loss_m) / count(loss_m), 1)         AS normalised_per_event_m,
       round(percentile_approx(loss_m, 0.5), 1)      AS median_event_m
FROM v_event_loss GROUP BY calendar_year ORDER BY calendar_year
""")

### 2.2 · Financial year with a 3-year moving average
Insurers plan in financial years (July–June). The moving average smooths out single bad years.

In [ ]:
q22 = olap("bq2_by_financial_year", """
WITH fy AS (
  SELECT au_financial_year AS fy, count(*) AS events, count(loss_m) AS events_with_loss,
         sum(loss_m) AS loss_m, sum(nominal_loss_m) AS nominal_m
  FROM v_event_loss GROUP BY au_financial_year)
SELECT fy, events, events_with_loss, round(nominal_m, 1) AS nominal_loss_m, round(loss_m, 1) AS normalised_loss_m,
       round(loss_m / events_with_loss, 1) AS normalised_per_event_m,
       round(avg(loss_m) OVER (ORDER BY fy ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1) AS loss_3yr_avg_m,
       round(sum(loss_m) OVER (ORDER BY fy ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
             / sum(events_with_loss) OVER (ORDER BY fy ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1) AS per_event_3yr_m
FROM fy ORDER BY fy
""")

### 2.3 · Drill-down into the worst calendar year
`ROLLUP (quarter, month)` inside one year gives the year total, each quarter, and each month in one result. The last query drills to the individual events.

In [ ]:
worst_year = int(q21.sort_values("normalised_loss_m", ascending=False).calendar_year.iloc[0])
print("Worst calendar year by normalised loss:", worst_year)

q23 = olap("bq2_drilldown_worst_year", f"""
SELECT CASE grouping_id(calendar_quarter, month_number) WHEN 3 THEN 'year' WHEN 1 THEN 'quarter' ELSE 'month' END AS level,
       {worst_year} AS calendar_year, calendar_quarter, month_number, CASE WHEN grouping(month_number) = 0 THEN first(month_name) END AS month_name,
       count(*) AS events, round(sum(loss_m), 1) AS loss_m
FROM v_event_loss WHERE calendar_year = {worst_year}
GROUP BY ROLLUP (calendar_quarter, month_number)
ORDER BY calendar_quarter NULLS FIRST, month_number NULLS FIRST
""")

q23e = olap("bq2_drilldown_worst_year_events", f"""
SELECT start_date, event_name, hazard_group, state_code, round(loss_m, 1) AS loss_m,
       round(100 * loss_m / sum(loss_m) OVER (), 1) AS pct_of_year
FROM v_event_loss WHERE calendar_year = {worst_year} ORDER BY loss_m DESC NULLS LAST
""")

### 2.4 · Did the recording threshold change?
If the source stopped listing small events, the **smallest recorded event per year** jumps up and the **event count** drops. That would make loss per event rise even if catastrophes did not get worse.

In [ ]:
q24 = olap("bq2_recording_threshold", """
SELECT calendar_year, count(*) AS events, count(nominal_loss_m) AS events_with_loss,
       round(min(nominal_loss_m), 1) AS smallest_event_m,
       round(percentile_approx(nominal_loss_m, 0.5), 1) AS median_event_m,
       count_if(nominal_loss_m < 100) AS events_under_100m
FROM v_event_loss GROUP BY calendar_year ORDER BY calendar_year
""")

### 2.5 · Like-for-like: only events of $100m or more
Slicing to events that would have been recorded in **every** year removes the threshold effect. Periods are three five-year blocks; 2026 is left out because the year is not complete.

In [ ]:
q25 = olap("bq2_like_for_like", """
SELECT period, count(*) AS large_events, round(count(*) / max(years), 1) AS large_events_per_year,
       round(sum(loss_m), 0) AS loss_m, round(sum(loss_m) / count(*), 0) AS loss_per_large_event_m,
       round(percentile_approx(loss_m, 0.5), 0) AS median_large_event_m,
       round(sum(loss_m) / max(years), 0) AS loss_per_year_m
FROM (
  SELECT CASE WHEN calendar_year <= 2015 THEN '2011–2015'
              WHEN calendar_year <= 2020 THEN '2016–2020'
              ELSE '2021–2025' END AS period,
         5 AS years, loss_m
  FROM v_event_loss WHERE loss_m >= 100 AND calendar_year <= 2025)
GROUP BY period ORDER BY period
""")